In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI , GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

c:\Users\TECH DRONA\Desktop\GEN_Ai\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
loader = PyPDFLoader("../data/YT_Script_Audio_PRO.pdf")
docs = loader.load()

In [5]:
len(docs)

3

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
spllited_docs = splitter.split_documents(docs)

In [7]:
len(spllited_docs)

6

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = InMemoryVectorStore.from_documents(
    documents=spllited_docs,
    embedding=embeddings
)


### agent - tools,llm , promt

In [ ]:
@tool
def retriever_tool(query:str):
    """
        this tool can help you to retrieve the relevent data of the pdf documents,
        and these documents have details about medical report
    """
    docs = vector_store.similarity_search(query=query, k=4)

    context = ""

    for doc in docs:
        context += doc.page_content + "\n\n"
    return context
    

In [16]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [17]:
system_prompt = """
    you are a helpful assistent that answers questions using retrived context.
    always use the  `retriever_tool` tool for the questions requiring external knowledge.

"""

In [18]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt     
    )

In [27]:
query = "what is best tool for get audio qualtiy like podcast"

responce = agent.invoke({'messages':[{"role":"user","content":query}]})

In [25]:
result = responce["messages"][-1].content

In [26]:
print(result)

I am sorry, I cannot answer this question. The available tool `retriever_tool` can only access information about medical reports. It cannot provide information about audio quality tools.
